# 07 — Jaringan jalan dengan OSMnx

OSMnx mengunduh jaringan jalan OpenStreetMap sebagai graf NetworkX yang topologinya sudah tersambung.
Satu baris untuk satu kota; lalu jenis jaringan, statistik graf, POI via Overpass, dan cek kualitas data.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import time

import geopandas as gpd
import osmnx as ox
import pandas as pd

from kota import CFG, OSM, PROC, RAW, UTM, ROOT

ox.settings.use_cache = True
ox.settings.cache_folder = str(ROOT / "data" / "cache" / "osmnx")

## 1. Satu baris, seluruh kota

In [ ]:
t0 = time.time()
try:
    G = ox.graph_from_place(CFG["osm_query"], network_type="drive")
    print(f"{G.number_of_nodes():,} simpul, {G.number_of_edges():,} ruas — {time.time() - t0:.0f} detik")
except Exception as e:  # server Overpass bersama bisa sibuk atau menolak sementara
    print(f"Overpass tidak merespons ({type(e).__name__}) — lanjut dengan snapshot di sel berikutnya")

Simpul adalah persimpangan dan ujung jalan; ruas adalah segmen jalan di antaranya. Graf jaringan `drive`
**berarah**: jalan dua arah disimpan sebagai dua ruas, jalan satu arah hanya satu.

Untuk kursus ini jaringan sudah dibekukan dalam `graphml`, supaya angka Anda sama dengan angka di video.

In [ ]:
G = ox.load_graphml(RAW / "jalan_drive.graphml")
W = ox.load_graphml(RAW / "jalan_walk.graphml")

## 2. Jenis jaringan: pilih dengan sengaja
`drive` mematuhi arah satu jalur dan tidak memuat gang serta jalan setapak. Analisis pejalan kaki di atas
jaringan `drive` memberi hasil keliru.

In [ ]:
def ringkas(g):
    e = ox.graph_to_gdfs(g, nodes=False)
    return {"simpul": g.number_of_nodes(), "ruas": g.number_of_edges(),
            "panjang (km)": round(e["length"].sum() / 1000),
            "ruas satu arah (%)": round(100 * e["oneway"].astype(bool).mean(), 1)}


pd.DataFrame({"drive": ringkas(G), "walk": ringkas(W)})

## 3. Statistik graf
Proyeksikan dulu ke UTM, lalu hitung statistik dasar. **Circuity** = panjang jaringan ÷ jarak garis lurus
antar-ujung ruas; makin tinggi, makin berkelok jaringannya.

In [ ]:
kota = gpd.read_file(OSM, layer="batas_kota").to_crs(UTM)
Gp = ox.project_graph(G, to_crs=UTM)
st = ox.basic_stats(Gp, area=kota.area.iloc[0])
pd.Series({
    "kepadatan persimpangan (/km²)": st["intersection_density_km"],
    "kepadatan jalan (km/km²)": st["street_density_km"] / 1000,
    "rerata jalan per simpul": st["streets_per_node_avg"],
    "circuity rata-rata": st["circuity_avg"],
}).round(3)

Bandingkan dua kecamatan: yang terkecil (biasanya pusat kota lama) dan yang terluas (biasanya pinggiran).
Dipilih dari data, bukan diketik, supaya notebook tetap jalan di kota lain — tetapi hanya di antara kecamatan
yang punya jaringan jalan: kecamatan kepulauan (misalnya Kepulauan Sangkarrang di Makassar) tidak punya.

In [ ]:
kec = gpd.read_file(OSM, layer="kecamatan")
kec["luas"] = kec.to_crs(UTM).area
simpul = ox.graph_to_gdfs(G, edges=False)[["geometry"]]
kec["n_simpul"] = kec["kecamatan"].map(gpd.sjoin(simpul, kec[["kecamatan", "geometry"]], predicate="within")
                                       .groupby("kecamatan").size()).fillna(0)
berjalan = kec[kec["n_simpul"] >= 100].sort_values("luas")
print("tanpa jaringan jalan:", list(kec.loc[kec["n_simpul"] < 100, "kecamatan"]) or "tidak ada")
pilihan = [berjalan.iloc[0]["kecamatan"], berjalan.iloc[-1]["kecamatan"]]
baris = {}
for nama in pilihan:
    poly = kec.loc[kec["kecamatan"] == nama, "geometry"].iloc[0]
    g = ox.truncate.truncate_graph_polygon(G, poly)
    luas = gpd.GeoSeries([poly], crs=4326).to_crs(UTM).area.iloc[0]
    s = ox.basic_stats(ox.project_graph(g, to_crs=UTM), area=luas)
    baris[nama] = {"luas (km²)": luas / 1e6, "persimpangan/km²": s["intersection_density_km"],
                   "km jalan/km²": s["street_density_km"] / 1000, "circuity": s["circuity_avg"]}
pd.DataFrame(baris).round(3)

## 4. POI dengan Overpass
`features_from_place` mengambil objek apa pun berdasarkan tag. Hati-hati dengan tag: banyak **posyandu**
ditandai `amenity=clinic`, sama seperti klinik dan puskesmas.

In [ ]:
poi = gpd.read_file(OSM, layer="poi")
klinik = poi[poi["amenity"] == "clinic"]
nama = klinik["name"].fillna("").str.lower()
pd.Series({
    "amenity=clinic": len(klinik),
    "  bernama 'posyandu'": int(nama.str.contains("posyandu").sum()),
    "  bernama 'puskesmas'": int(nama.str.contains("puskesmas").sum()),
    "  tanpa nama": int((nama == "").sum()),
    "amenity=school": int((poi["amenity"] == "school").sum()),
    "amenity=marketplace": int((poi["amenity"] == "marketplace").sum()),
}, name="jumlah")

In [ ]:
puskesmas = poi[poi["name"].fillna("").str.contains("puskesmas", case=False)
                & ~poi["name"].fillna("").str.contains("pembantu|pustu", case=False)]
puskesmas = puskesmas.drop_duplicates("name").to_crs(UTM)
puskesmas[["name", "amenity", "healthcare", "geometry"]].to_file(PROC / "puskesmas.gpkg", layer="puskesmas")
print(len(puskesmas), "puskesmas (tanpa pustu) disimpan")

## 5. Kualitas data
Kelengkapan OSM sangat berbeda antar-kota di Indonesia. Jaringan yang jarang bisa berarti pemetaannya jarang,
bukan jalannya. Satu indikator sederhana: bagian ruas jalan yang bernama, per kecamatan.

In [ ]:
e = ox.graph_to_gdfs(W, nodes=False)[["name", "length", "geometry"]].reset_index(drop=True)
e["bernama"] = e["name"].notna()
e = gpd.sjoin(e.set_geometry(e.geometry.representative_point()), kec[["kecamatan", "geometry"]],
              predicate="within")
q = e.groupby("kecamatan").apply(lambda d: 100 * d.loc[d.bernama, "length"].sum() / d["length"].sum(),
                                 include_groups=False)
q.sort_values().round(1).rename("% panjang jalan bernama")

In [ ]:
fig, ax = ox.plot_graph(Gp, node_size=0, edge_linewidth=0.3, bgcolor="white", edge_color="#333", show=False)
ax.set_title(f"Jaringan jalan drive — {CFG['name']}");